# S&P 500 price-vs-30-day-SMA timing rule -- backtest

**Rule** (price-versus-moving-average, *not* a 50/200 crossover):

1. Compute the 30-trading-day simple moving average of the close:
   $SMA_{30,t} = \frac{1}{30}\sum_{i=0}^{29} P_{t-i}$
2. **Risk-on** -- close above the SMA: hold the S&P 500 ETF.
3. **Risk-off** -- close below the SMA: sell the ETF and hold T-bills.
4. Decide **after the close**, trade at the **next session's open** (no look-ahead).

**Implementation**

| Item | Choice |
|---|---|
| Signal | split-adjusted close vs its SMA, known at the close of day *t* |
| Rebalancing | `REBALANCE`: `daily` checks the rule every close (the original rule); `weekly` / `monthly` / `quarterly` / `yearly` check it only at the **last close of each period** and hold the position in between |
| Execution | default `next_open`: decided at close *t*, filled at open *t+1*; returns measured open-to-open |
| Equity returns | total return (dividends reinvested): the `Adj Close / Close` factor is applied to the open too |
| Risk-off asset | `irx`: accrual at the 13-week T-bill yield (`^IRX` index, not tradable); `bil`: SPDR Bloomberg 1-3 Month T-Bill ETF (2007+, `^IRX` before); `zero`: plain cash |
| Costs | `COST_BPS` charged per switch (the whole portfolio changes hands) |

> Decision-support / research only -- this notebook never places orders. Past performance is not indicative of future results.

In [1]:
# ---- Parameters -------------------------------------------------------------
EQUITY_TICKER = "SPY"    # investable S&P 500 ETF; "^GSPC" = price-only index (no dividends, unreliable opens pre-1962)
CASH_SOURCE = "irx"      # "irx" | "bil" | "zero"
SMA_WINDOW = 200
REBALANCE = "monthly"    # "daily" (original rule) | "weekly" | "monthly" | "quarterly" | "yearly"
START = "1995-01-01"     # first backtest date (SMA warm-up comes from earlier data)
END = None               # None = latest available
COST_BPS = 5.0           # cost per switch, bp of portfolio value (spread + commission + slippage)
EXECUTION = "next_open"  # "next_open" (the rule) | "next_close" | "same_close" (look-ahead biased, comparison only)
REFRESH_DATA = False     # True = ignore the local cache and re-download
CACHE_MAX_AGE_HOURS = 12

In [2]:
import math
import sys
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# Shared helpers: notebooks/backtest/bt_common.py
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
from bt_common import (  # noqa: E402
    REBALANCE_FREQ, EXECUTIONS, REPO_ROOT, SESSION, cash_returns, perf_stats, setup_plotly, sma_backtest, strategy_stats, total_return_frame,
    trade_log,
)
from bt_common import load_history as _load_history  # noqa: E402

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

C_STRAT, C_BH, C_CASH, C_SMA = "#1f77b4", "#ff7f0e", "#2ca02c", "#333333"

# fig.show() emits plain HTML, so charts render in VS Code without the "Jupyter Notebook Renderers" extension.
PLOTLY_JS = "cdn"  # "cdn" = load plotly.js from cdn.plot.ly | "inline" = embed it (offline, ~4.6 MB per chart)
setup_plotly(PLOTLY_JS)
load_history = partial(_load_history, refresh=REFRESH_DATA, max_age_hours=CACHE_MAX_AGE_HOURS)

print("repo root:", REPO_ROOT)
print("yfinance session:", type(SESSION).__name__ if SESSION is not None else "default")

repo root: C:\Users\wn686\OneDrive - Deutsche Börse AG\Desktop\REPOs\investment-firm-agents
yfinance session: Session


## 1. Data

In [3]:
px_all = total_return_frame(load_history(EQUITY_TICKER))
px_all = px_all[px_all["open"] > 0]
if END is not None:
    px_all = px_all.loc[:END]

irx = load_history("^IRX")["Close"].rename("irx_pct")
bil_px = total_return_frame(load_history("BIL")) if CASH_SOURCE == "bil" else None

print(f"{EQUITY_TICKER}: {px_all.index[0].date()} -> {px_all.index[-1].date()}  ({len(px_all):,} bars)")
print(f"^IRX: {irx.index[0].date()} -> {irx.index[-1].date()}, last = {irx.iloc[-1]:.2f}%")
px_all.tail()

SPY: 1993-01-29 -> 2026-10-19  (8,488 bars)
^IRX: 1960-01-04 -> 2026-10-19, last = 4.08%


,open,close,tr_open,tr_close
Date,,,,
2026-10-13,777.340027,778.010010,777.340027,778.010010
2026-10-14,774.919983,769.599976,774.919983,769.599976
2026-10-15,766.250000,771.789978,766.250000,771.789978
2026-10-16,773.239990,763.250000,773.239990,763.250000
2026-10-19,767.229980,767.380005,767.229980,767.380005


In [4]:
CASH = {f: cash_returns(px_all.index, f, CASH_SOURCE, irx, bil_px) for f in ("tr_open", "tr_close")}

## 2. Backtest engine

**Rebalancing schedule.** With `REBALANCE = "daily"` the rule is checked at every close. Otherwise it is checked only on the **last trading day of each week / month / quarter / year**. If the close is then on the other side of the SMA than the current position, the portfolio switches at the next open; between check dates the position is held regardless of what the price does. So there is at most one switch per period.

**Timing** for `next_open` (row *t* = holding interval open *t-1* -> open *t*):

* the target $s_{t-2}$ (signal at the last check date up to *t-2*) is known at the close of *t-2*,
* the order fills at the open of *t-1*,
* the position earns the open-to-open return to *t*. Hence `position = target.shift(2)`.

`same_close` (`target.shift(1)` on close-to-close returns) trades at the same close that produced the signal. That is the look-ahead the rule warns about, so it is shown only for comparison.

In [5]:
def backtest(px, window=30, cost_bps=0.0, execution="next_open", start=None, rebalance="daily") -> pd.DataFrame:
    # Shared engine (bt_common.sma_backtest) with this notebook's T-bill series.
    return sma_backtest(px, CASH, window, cost_bps, execution, start, rebalance)


res = backtest(px_all, SMA_WINDOW, COST_BPS, EXECUTION, start=START, rebalance=REBALANCE)
trades = trade_log(res, px_all, EXECUTION)
print(f"Backtest {res.index[0].date()} -> {res.index[-1].date()} | SMA{SMA_WINDOW} | rebalance {REBALANCE} | "
      f"{EXECUTION} | {COST_BPS} bp/switch | cash={CASH_SOURCE}")

Backtest 1995-01-03 -> 2026-10-19 | SMA200 | rebalance monthly | next_open | 5.0 bp/switch | cash=irx


## 3. Results

In [6]:
summary = pd.DataFrame(
    {
        f"SMA{SMA_WINDOW} {REBALANCE} (net)": perf_stats(res["r_strat"], res["r_cash"]),
        f"SMA{SMA_WINDOW} {REBALANCE} (gross)": perf_stats(res["r_gross"], res["r_cash"]),
        f"Buy & hold {EQUITY_TICKER}": perf_stats(res["r_bh"], res["r_cash"]),
        "T-bills": perf_stats(res["r_cash"], res["r_cash"]),
    }
)
PCT_ROWS = {"CAGR", "Vol", "Max DD", "Worst day"}


def _fmt(row, v):
    if isinstance(v, float) and not pd.isna(v):
        return f"{v:.2%}" if row in PCT_ROWS else f"{v:.2f}"
    return v


display(summary.apply(lambda col: [_fmt(r, v) for r, v in col.items()]).set_axis(summary.index))

ss = strategy_stats(res)
print(f"Time in market {ss['Time in market']:.1%} | switches {ss['Switches']} "
      f"({ss['Switches / yr']:.1f}/yr) | cost drag {ss['Cost drag / yr']:.2%}/yr")

,SMA200 monthly (net),SMA200 monthly (gross),Buy & hold SPY,T-bills
CAGR,10.92%,10.99%,11.15%,2.39%
Vol,12.78%,12.78%,18.80%,0.18%
Sharpe,0.69,0.70,0.53,NaN
Sortino,0.97,0.97,0.75,NaN
Max DD,-25.41%,-25.18%,-55.42%,-0.00%
Calmar,0.43,0.44,0.20,NaN
Worst day,-9.29%,-9.29%,-12.94%,-0.00%
MaxDD peak,2022-01-04,2022-01-04,2007-10-11,2020-03-20
MaxDD recovered,2024-06-18,2024-06-13,2012-08-17,2020-04-13
Final x,27.01,27.54,28.82,2.12


Time in market 75.9% | switches 39 (1.2/yr) | cost drag 0.06%/yr


In [7]:
curves = pd.DataFrame(
    {
        f"SMA{SMA_WINDOW} {REBALANCE} net": (1 + res["r_strat"]).cumprod(),
        f"Buy & hold {EQUITY_TICKER}": (1 + res["r_bh"]).cumprod(),
        "T-bills": (1 + res["r_cash"]).cumprod(),
    }
)
dd = curves / curves.cummax() - 1
# The SMA rescaled onto the buy & hold curve: B&H is above this line exactly when close > SMA.
sma_on_bh = res["sma"] / res["close"] * curves.iloc[:, 1]
w_eq = res["pos"]  # SPY weight held over each interval (0 or 1); T-bill weight = 1 - w_eq

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, row_heights=[0.55, 0.17, 0.28], vertical_spacing=0.05,
    subplot_titles=(f"Growth of $1 (log scale), {res.index[0].date()} -> {res.index[-1].date()}",
                    "Strategy weights", "Drawdown"),
)
for name, color in zip(curves.columns, [C_STRAT, C_BH, C_CASH]):
    fig.add_trace(go.Scatter(x=curves.index, y=curves[name], name=name, line=dict(color=color, width=1.4),
                             legendgroup="growth"), row=1, col=1)
fig.add_trace(go.Scatter(x=sma_on_bh.index, y=sma_on_bh, name=f"SMA{SMA_WINDOW} of {EQUITY_TICKER}",
                         line=dict(color=C_SMA, width=0.8), opacity=0.7, legendgroup="growth"), row=1, col=1)

fig.add_trace(go.Scatter(x=w_eq.index, y=w_eq, name=f"{EQUITY_TICKER} weight", line_shape="hv",
                         line=dict(color=C_BH, width=0), fill="tozeroy", fillcolor="rgba(255,127,14,0.45)",
                         legendgroup="w"), row=2, col=1)
fig.add_trace(go.Scatter(x=w_eq.index, y=np.ones(len(w_eq)), name="T-bill weight", line_shape="hv",
                         line=dict(color=C_CASH, width=0), fill="tonexty", fillcolor="rgba(44,160,44,0.4)",
                         legendgroup="w", hoverinfo="skip"), row=2, col=1)
fig.add_trace(go.Scatter(x=w_eq.index, y=w_eq.rolling(252).mean(), name=f"{EQUITY_TICKER} weight, 1y avg",
                         line=dict(color="black", width=1), legendgroup="w"), row=2, col=1)

for name, color in zip(curves.columns[:2], [C_STRAT, C_BH]):
    fig.add_trace(go.Scatter(x=dd.index, y=dd[name], name=f"DD {name}", line=dict(color=color, width=1),
                             fill="tozeroy", legendgroup="dd"), row=3, col=1)

fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(tickformat=".0%", range=[0, 1], row=2, col=1)
fig.update_yaxes(tickformat=".0%", row=3, col=1)
fig.update_layout(height=900, hovermode="x unified", legend=dict(orientation="h", y=-0.06))
fig.show()

In [8]:
# Last 2 years: price vs SMA, check dates, fills, risk-off periods shaded
years = 10
recent = res.loc[res.index[-1] - pd.DateOffset(years=years):]
fig = go.Figure()
fig.add_trace(go.Scatter(x=recent.index, y=recent["close"], name=f"{EQUITY_TICKER} close", line=dict(color=C_BH)))
fig.add_trace(go.Scatter(x=recent.index, y=recent["sma"], name=f"SMA{SMA_WINDOW}", line=dict(color=C_SMA, width=1)))
if REBALANCE != "daily":
    chk = recent[recent["check"]]
    fig.add_trace(go.Scatter(x=chk.index, y=chk["close"], mode="markers", name=f"{REBALANCE} check (close)",
                             marker=dict(symbol="circle-open", size=8, color="grey")))
tr = trades[trades["filled"] >= recent.index[0]]
for action, sym, col in (("BUY equity", "triangle-up", "green"), ("SELL -> T-bills", "triangle-down", "red")):
    t = tr[tr["action"] == action]
    fig.add_trace(go.Scatter(x=t["filled"], y=t["fill price"], mode="markers", name=action,
                             marker=dict(symbol=sym, size=11, color=col)))
off = (recent["pos"] == 0).astype(int)
grp = (off != off.shift()).cumsum()
for _, g in recent[off == 1].groupby(grp[off == 1]):
    fig.add_vrect(x0=g.index[0], x1=g.index[-1], fillcolor="red", opacity=0.08, line_width=0, layer="below")
fig.update_layout(title=f"Last {years} years -- shaded = holding T-bills (rebalance {REBALANCE})", height=500,
                  hovermode="x unified", legend=dict(orientation="h", y=-0.15))
fig.show()

In [9]:
yearly = pd.DataFrame(
    {
        f"SMA{SMA_WINDOW} net": (1 + res["r_strat"]).groupby(res.index.year).prod() - 1,
        "Buy & hold": (1 + res["r_bh"]).groupby(res.index.year).prod() - 1,
        "T-bills": (1 + res["r_cash"]).groupby(res.index.year).prod() - 1,
    }
)
yearly["Excess vs B&H"] = yearly.iloc[:, 0] - yearly.iloc[:, 1]
yearly["Switches"] = (res["turnover"] > 0).groupby(res.index.year).sum()

fig = go.Figure()
for name, color in zip(yearly.columns[:2], [C_STRAT, C_BH]):
    fig.add_trace(go.Bar(x=yearly.index, y=yearly[name], name=name, marker_color=color))
fig.update_layout(title="Calendar-year returns", barmode="group", yaxis_tickformat=".0%", height=420)
fig.show()
display(yearly.style.format({c: "{:.1%}" for c in yearly.columns if c != "Switches"}))

,SMA200 net,Buy & hold,T-bills,Excess vs B&H,Switches
Date,,,,,
1995,33.9%,36.1%,5.5%,-2.2%,1
1996,24.9%,24.9%,5.0%,0.0%,0
1997,30.7%,30.7%,5.0%,0.0%,0
1998,12.1%,28.9%,4.8%,-16.9%,2
1999,13.2%,20.4%,4.6%,-7.2%,2
2000,0.3%,-7.7%,5.8%,8.1%,1
2001,3.4%,-12.3%,3.4%,15.7%,0
2002,1.6%,-23.0%,1.6%,24.6%,0
2003,22.9%,28.5%,1.0%,-5.6%,1


In [10]:
# Rolling 3-year annualised return: strategy minus buy & hold
WIN = 252 * 3


def roll_ann(r):
    return np.exp(np.log1p(r).rolling(WIN).sum() * 252 / WIN) - 1


rel = (roll_ann(res["r_strat"]) - roll_ann(res["r_bh"])).dropna()
fig = go.Figure()
fig.add_trace(go.Scatter(x=rel.index, y=rel.clip(lower=0), fill="tozeroy", name="strategy ahead",
                         line=dict(color="green", width=0.5)))
fig.add_trace(go.Scatter(x=rel.index, y=rel.clip(upper=0), fill="tozeroy", name="strategy behind",
                         line=dict(color="red", width=0.5)))
fig.update_layout(title="Rolling 3-year annualised return: strategy minus buy & hold", yaxis_tickformat=".0%",
                  height=380, hovermode="x unified")
fig.show()
print(f"Strategy beat buy & hold in {(rel > 0).mean():.0%} of rolling 3-year windows")

Strategy beat buy & hold in 29% of rolling 3-year windows


## 4. Trades and current signal

In [11]:
def spells(df: pd.DataFrame) -> pd.DataFrame:
    # Contiguous risk-on / risk-off periods and what each leg earned during them.
    grp = (df["pos"] != df["pos"].shift()).cumsum()
    out = df.groupby(grp).agg(
        start=("pos", lambda s: s.index[0].date()),
        end=("pos", lambda s: s.index[-1].date()),
        state=("pos", "first"),
        days=("pos", "size"),
        equity_ret=("r_eq", lambda r: (1 + r).prod() - 1),
        cash_ret=("r_cash", lambda r: (1 + r).prod() - 1),
    )
    out["state"] = out["state"].map({1.0: "risk-on", 0.0: "risk-off"})
    return out.reset_index(drop=True)


sp = spells(res)
on, off = sp[sp["state"] == "risk-on"], sp[sp["state"] == "risk-off"]
print(f"Risk-on spells: {len(on)}, median {on['days'].median():.0f} days, "
      f"equity beat T-bills in {(on['equity_ret'] > on['cash_ret']).mean():.0%} of them")
print(f"Risk-off spells: {len(off)}, median {off['days'].median():.0f} days, "
      f"equity lagged T-bills (loss avoided) in {(off['equity_ret'] < off['cash_ret']).mean():.0%} of them")
print(f"Spells shorter than 5 days (whipsaws): {(sp['days'] < 5).mean():.0%}")
display(trades.assign(filled=trades["filled"].dt.date).tail(10))

Risk-on spells: 20, median 230 days, equity beat T-bills in 80% of them
Risk-off spells: 20, median 43 days, equity lagged T-bills (loss avoided) in 15% of them
Spells shorter than 5 days (whipsaws): 0%


,filled,action,fill price
29,2022-05-02,SELL -> T-bills,412.07
30,2022-12-01,BUY equity,408.77
31,2023-01-03,SELL -> T-bills,384.37
32,2023-02-01,BUY equity,405.21
33,2023-11-01,SELL -> T-bills,419.20
34,2023-12-01,BUY equity,455.77
35,2025-04-01,SELL -> T-bills,557.45
36,2025-06-02,BUY equity,587.76
37,2026-04-01,SELL -> T-bills,653.90
38,2026-05-01,BUY equity,721.25


In [12]:
last_day = px_all.index[-1]
last_close = px_all["close"].iloc[-1]
last_sma = px_all["close"].iloc[-SMA_WINDOW:].mean()
rule = "above" if last_close > last_sma else "below"
target = "equity" if res["target"].iloc[-1] == 1 else "T-bills"
print(f"As of close {last_day.date()}: {EQUITY_TICKER} {last_close:.2f} is {rule} SMA{SMA_WINDOW} {last_sma:.2f} "
      f"({last_close / last_sma - 1:+.2%}).")
if REBALANCE == "daily":
    print(f"Daily rule -> hold {target} from the next open.")
else:
    freq = REBALANCE_FREQ[REBALANCE]
    per = last_day.to_period(freq)
    is_check = bool(res["check"].iloc[-1])
    nxt = per + 1 if is_check else per
    next_check = pd.offsets.BDay().rollback(nxt.end_time.normalize())
    when = "today is a check date" if is_check else "today is not a check date"
    print(f"{REBALANCE.capitalize()} rebalancing, {when} -> hold {target} from the next open; "
          f"next check at the close of ~{next_check.date()} (holidays ignored).")
print("Research output only -- not a recommendation or an order.")

As of close 2026-10-19: SPY 767.38 is above SMA200 725.44 (+5.78%).
Monthly rebalancing, today is not a check date -> hold equity from the next open; next check at the close of ~2026-10-30 (holidays ignored).
Research output only -- not a recommendation or an order.


## 5. Robustness

* **Rebalancing frequency** -- does checking less often reduce whipsaws enough to pay for the slower reaction?
* **Execution timing** -- how much of the result comes from look-ahead?
* **Window sweep** -- is 30 days special or one point on a smooth curve? (Picking the best window in-sample is overfitting.)
* **Costs** -- the rule can switch often, so costs matter.
* **Sub-periods** -- where the edge comes from.

In [13]:
FMT = {"CAGR": "{:.2%}", "Sharpe": "{:.2f}", "Max DD": "{:.1%}", "Switches / yr": "{:.1f}", "Time in market": "{:.0%}"}


def row(b):
    s, st = perf_stats(b["r_strat"], b["r_cash"]), strategy_stats(b)
    return {"CAGR": s["CAGR"], "Sharpe": s["Sharpe"], "Max DD": s["Max DD"],
            "Switches / yr": st["Switches / yr"], "Time in market": st["Time in market"]}


bh_s = perf_stats(res["r_bh"], res["r_cash"])
reb = {f: row(backtest(px_all, SMA_WINDOW, COST_BPS, EXECUTION, start=START, rebalance=f)) for f in REBALANCE_FREQ}
reb["buy & hold"] = {"CAGR": bh_s["CAGR"], "Sharpe": bh_s["Sharpe"], "Max DD": bh_s["Max DD"],
                     "Switches / yr": 0.0, "Time in market": 1.0}
print(f"SMA{SMA_WINDOW}, {EXECUTION}, {COST_BPS} bp/switch -- by rebalancing frequency")
display(pd.DataFrame(reb).T.style.format(FMT))

exe = {ex: row(backtest(px_all, SMA_WINDOW, COST_BPS, ex, start=START, rebalance=REBALANCE)) for ex in EXECUTIONS}
print(f"SMA{SMA_WINDOW}, rebalance {REBALANCE} -- by execution timing")
display(pd.DataFrame(exe).T.style.format(FMT))

SMA200, next_open, 5.0 bp/switch -- by rebalancing frequency


,CAGR,Sharpe,Max DD,Switches / yr,Time in market
daily,9.02%,0.58,-25.1%,6.6,76%
weekly,8.81%,0.56,-27.4%,3.4,75%
monthly,10.92%,0.69,-25.4%,1.2,76%
quarterly,8.63%,0.51,-32.0%,0.9,74%
yearly,9.13%,0.52,-38.9%,0.5,69%
buy & hold,11.15%,0.53,-55.4%,0.0,100%


SMA200, rebalance monthly -- by execution timing


,CAGR,Sharpe,Max DD,Switches / yr,Time in market
next_open,10.92%,0.69,-25.4%,1.2,76%
next_close,10.94%,0.70,-26.3%,1.2,76%
same_close,11.20%,0.72,-24.3%,1.2,76%


In [14]:
WINDOWS = [5, 10, 20, 30, 50, 75, 100, 150, 200, 250, 300, 350, 400]
grid = pd.DataFrame(
    {f: {w: perf_stats(b["r_strat"], b["r_cash"])["Sharpe"]
         for w in WINDOWS
         for b in [backtest(px_all, w, COST_BPS, EXECUTION, start=START, rebalance=f)]}
     for f in REBALANCE_FREQ}
)
fig = go.Figure(go.Heatmap(z=grid.values, x=grid.columns, y=[str(w) for w in grid.index], colorscale="RdYlGn",
                           text=grid.round(2).values, texttemplate="%{text}", colorbar=dict(title="Sharpe")))
fig.update_layout(title=f"Sharpe ratio by SMA window x rebalancing frequency ({COST_BPS} bp/switch); "
                        f"buy & hold = {bh_s['Sharpe']:.2f}",
                  yaxis_title="SMA window (days)", xaxis_title="rebalancing", height=480)
fig.show()

In [15]:
COSTS = [0.0, 5.0, 20.0]
recs = []
for w in WINDOWS:
    for c in COSTS:
        b = backtest(px_all, w, c, EXECUTION, start=START, rebalance=REBALANCE)
        recs.append({"window": w, "cost_bp": c, **row(b)})
sweep = pd.DataFrame(recs)

fig = make_subplots(rows=1, cols=3, subplot_titles=("CAGR", "Sharpe", "Max DD"))
palette = {0.0: "#1f77b4", 5.0: "#ff7f0e", 20.0: "#d62728"}
for i, metric in enumerate(["CAGR", "Sharpe", "Max DD"], start=1):
    for c in COSTS:
        s = sweep[sweep["cost_bp"] == c]
        fig.add_trace(go.Scatter(x=s["window"], y=s[metric], mode="lines+markers", name=f"{c:.0f} bp",
                                 line=dict(color=palette[c]), legendgroup=str(c), showlegend=i == 1), row=1, col=i)
    fig.add_hline(y=bh_s[metric], line_dash="dash", line_color="black", row=1, col=i,
                  annotation_text="buy & hold" if i == 1 else None)
    fig.add_vline(x=SMA_WINDOW, line_dash="dot", line_color="grey", row=1, col=i)
    if metric != "Sharpe":
        fig.update_yaxes(tickformat=".0%", row=1, col=i)
fig.update_xaxes(title_text="SMA window (days)")
fig.update_layout(title=f"Window sweep (rebalance {REBALANCE}, by cost per switch)", height=420)
fig.show()
display(sweep[sweep["cost_bp"] == COST_BPS].drop(columns="cost_bp").set_index("window").style.format(FMT))

,CAGR,Sharpe,Max DD,Switches / yr,Time in market
window,,,,,
5,5.59%,0.31,-44.7%,6.1,55%
10,5.58%,0.31,-39.6%,5.7,58%
20,5.07%,0.27,-31.3%,6.2,61%
30,5.34%,0.30,-36.3%,5.5,64%
50,6.45%,0.38,-30.2%,4.3,65%
75,7.51%,0.47,-25.5%,3.4,69%
100,8.62%,0.55,-25.9%,2.8,71%
150,9.86%,0.63,-22.7%,1.9,74%
200,10.92%,0.69,-25.4%,1.2,76%


In [16]:
PERIODS = {
    "1993-1999 bull": ("1993-01-01", "1999-12-31"),
    "2000-2002 dot-com bust": ("2000-03-24", "2002-10-09"),
    "2002-2007 recovery": ("2002-10-10", "2007-10-09"),
    "2007-2009 GFC": ("2007-10-10", "2009-03-09"),
    "2009-2019 bull": ("2009-03-10", "2019-12-31"),
    "2020 COVID crash": ("2020-02-19", "2020-03-23"),
    "2020-2021 rebound": ("2020-03-24", "2021-12-31"),
    "2022 bear": ("2022-01-03", "2022-10-12"),
    "2023-latest": ("2023-01-01", None),
}
recs = {}
for name, (a, b) in PERIODS.items():
    if pd.Timestamp(a) < res.index[0]:
        continue  # period starts before the backtest
    sl = res.loc[a:b]
    if len(sl) < 5:
        continue
    recs[name] = {
        "strategy": (1 + sl["r_strat"]).prod() - 1,
        "buy & hold": (1 + sl["r_bh"]).prod() - 1,
        "time in market": sl["pos"].mean(),
        "switches": int((sl["turnover"] > 0).sum()),
    }
display(pd.DataFrame(recs).T.style.format(
    {"strategy": "{:+.1%}", "buy & hold": "{:+.1%}", "time in market": "{:.0%}", "switches": "{:.0f}"}))

,strategy,buy & hold,time in market,switches
2000-2002 dot-com bust,+3.2%,-45.3%,21%,1
2002-2007 recovery,+73.9%,+115.0%,82%,5
2007-2009 GFC,-4.1%,-55.0%,16%,1
2009-2019 bull,+193.5%,+489.5%,83%,13
2020 COVID crash,-11.4%,-31.8%,38%,1
2020-2021 rebound,+60.7%,+113.9%,89%,1
2022 bear,-16.2%,-23.8%,31%,3
2023-latest,+60.7%,+111.3%,89%,8


## 6. Caveats

* **Data** -- Yahoo Finance via `yfinance` (unofficial). Dividends come from the `Adj Close` factor, and SPY's expense ratio is already in its prices. Taxes on frequent switches are ignored.
* **T-bill proxy** -- `^IRX` is a discount yield applied as calendar-day accrual, so it differs slightly from a real T-bill or ETF return (bid/ask, fees). `bil` uses the actual ETF where available.
* **Costs** -- a flat `COST_BPS` per switch. Opening-auction slippage on volatile days can be higher.
* **Gap risk** -- the signal is known at the close but filled at the next open, so the strategy bears overnight gaps. This delay is what removes the look-ahead.
* **Rebalancing calendar** -- results for weekly/monthly/... schedules depend on *which* day is the check date (here: the last trading day of the period). A different anchor day can give noticeably different numbers.
* **Overfitting** -- the sweeps are descriptive only. Picking the best window or frequency in-sample overstates expected performance.